# Slovenian Spoken-Written UD Treebank: Dependency Arc Classification Demo

## What This Notebook Does

This notebook demonstrates **dependency arc classification** from the Slovenian Universal Dependencies (UD) treebank pair:
- **sl_sst**: spoken language (6,121 sentences)
- **sl_ssj**: written language (13,435 sentences)

Each dependency arc is classified into one of three categories:
- **ARGUMENT** (nsubj, obj, ccomp, xcomp, csubj): core roles required by the predicate
- **ADJUNCT** (advcl, acl, acl:relcl): optional adjuncts and clauses
- **MODIFIER** (nmod, amod, advmod): noun and adverbial modifiers

The dataset contains 128,162 classified dependency arcs with morphological and distance features.

**Dataset characteristics:**
- **Language**: Slovenian (sl)
- **Modalities**: Spoken vs. Written
- **Morphological richness**: case_richness = 0.9406 (76,890 / 81,750 nominals marked for case)
- **UD version**: v2.17
- **Source**: commul/universal_dependencies (HuggingFace Hub)


## Install Dependencies

This cell installs all required packages. On Colab, pre-installed core packages (numpy, pandas, matplotlib) are skipped to avoid ABI corruption. Locally, they are installed at Colab's exact versions.

In [ ]:
import subprocess, sys
def _pip(*a):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Non-Colab packages (always install)
_pip('loguru==0.7.2')

# Core packages (pre-installed on Colab, install locally only)
if 'google.colab' not in sys.modules:
    _pip(
        'numpy==2.0.2',
        'pandas==2.2.2',
        'matplotlib==3.10.0',
        'seaborn==0.13.2'
    )

print('✓ Dependencies installed')

## Imports

Import all required modules for data loading, analysis, and visualization.

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from loguru import logger

# Configure logging
logger.remove()
logger.add(sys.stdout, level='INFO', format='{time:HH:mm:ss}|{level:<7}|{message}')

print('✓ All imports successful')

## Configuration

Define tunable parameters. These are set to **minimal values** for fast prototyping. To use the full dataset, uncomment the `# FULL PARAMETERS` lines.

In [ ]:
# === MINIMAL PARAMETERS (for demo) ===
MAX_EXAMPLES = 100  # Process first N examples from dataset
# FULL PARAMETERS:
# MAX_EXAMPLES = None  # Process all examples (128,162 for Slovenian)

# Data loading
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-033e16-argument-adjunct-asymmetry-in-spoken-reg/main/round-1/dataset-1/demo/mini_demo_data.json"

# Visualization
PLOT_FIGSIZE = (12, 6)
PLOT_DPI = 100

logger.info(f'Config: MAX_EXAMPLES={MAX_EXAMPLES}')

## Data Loading

Load dataset from GitHub URL with fallback to local file. This pattern ensures the notebook works both locally (before GitHub deployment) and on Colab (after deployment).

In [ ]:
def load_data():
    """Load mini_demo_data.json from GitHub or local fallback."""
    try:
        import urllib.request
        logger.info(f'Trying to load from GitHub: {GITHUB_DATA_URL}')
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            data = json.loads(response.read().decode())
            logger.info(f'✓ Loaded {len(data["datasets"][0]["examples"])} examples from GitHub')
            return data
    except Exception as e:
        logger.debug(f'GitHub load failed: {e}')
    
    # Fallback to local file
    local_path = Path('mini_demo_data.json')
    if local_path.exists():
        logger.info(f'Loading from local file: {local_path}')
        with open(local_path) as f:
            data = json.load(f)
        logger.info(f'✓ Loaded {len(data["datasets"][0]["examples"])} examples from local file')
        return data
    
    raise FileNotFoundError('Could not load mini_demo_data.json from GitHub or local file')

data = load_data()

## Parse and Explore Data Structure

Extract metadata and prepare examples for analysis. Each example represents one classified dependency arc with pipe-delimited features in the 'input' field and the relation category in the 'output' field.

In [ ]:
# Extract metadata
metadata = data['metadata']
logger.info(f'Source: {metadata["source"]}')
logger.info(f'UD version: {metadata["ud_version"]}')
logger.info(f'Treebanks: {metadata["treebanks"]}')

# Extract relation category definitions
relation_categories = metadata['relation_categories']
logger.info(f'Relation categories defined: {list(relation_categories.keys())}')

# Extract case richness index
case_richness_index = metadata['case_richness_per_language']['sl']['case_richness_index']
logger.info(f'Slovenian case richness index: {case_richness_index:.4f}')

# Extract all examples
all_examples = []
for dataset_group in data['datasets']:
    group_name = dataset_group['dataset']
    examples = dataset_group['examples']
    logger.info(f'Dataset group "{group_name}": {len(examples)} total examples')
    all_examples.extend(examples)

# Limit to MAX_EXAMPLES if set
if MAX_EXAMPLES is not None:
    all_examples = all_examples[:MAX_EXAMPLES]
    logger.info(f'Limiting to first {MAX_EXAMPLES} examples for demo')

logger.info(f'\nProcessing {len(all_examples)} examples')

## Extract Features and Build DataFrame

Parse the pipe-delimited 'input' field to extract individual features:
- language: 'sl' (Slovenian)
- modality: 'spoken' or 'written'
- deprel: dependency relation label
- upos: part-of-speech tag
- dependency_distance: |head_pos - dep_pos|
- sentence_length: count of non-PUNCT tokens
- case_richness: morphological richness index

In [ ]:
def parse_input_field(input_str):
    """Parse pipe-delimited input string into feature dict."""
    parts = input_str.split('|')
    features = {}
    features['language'] = parts[0]
    features['modality'] = parts[1]
    features['deprel'] = parts[2]
    features['upos'] = parts[3]
    
    # Parse remaining key=value pairs
    for part in parts[4:]:
        if '=' in part:
            k, v = part.split('=')
            try:
                features[k] = float(v)
            except ValueError:
                features[k] = v
    
    return features

# Build DataFrame
rows = []
for ex in all_examples:
    input_features = parse_input_field(ex['input'])
    row = {
        'language': input_features['language'],
        'modality': input_features['modality'],
        'deprel': input_features['deprel'],
        'upos': input_features['upos'],
        'dependency_distance': int(input_features['dist']),
        'sentence_length': int(input_features['len']),
        'case_richness': input_features['cr'],
        'relation_category': ex['output'],
        'treebank': ex['metadata_treebank'],
        'sentence_id': ex['metadata_sentence_id'],
    }
    rows.append(row)

df = pd.DataFrame(rows)
logger.info(f'\nDataFrame created: {len(df)} rows, {len(df.columns)} columns')
logger.info(f'Columns: {list(df.columns)}')

## Summary Statistics

Compute basic statistics over the dataset to understand distribution across modalities, relation categories, and linguistic features.

In [ ]:
# Display first few rows
logger.info('\n=== First 5 Examples ===')
print(df.head())

# Summary by relation category
logger.info('\n=== Relation Category Distribution ===')
rel_counts = df['relation_category'].value_counts()
print(rel_counts)
print(f'\nTotal: {rel_counts.sum()} examples')

# Summary by modality
logger.info('\n=== Distribution by Modality ===')
modality_counts = df.groupby('modality').size()
print(modality_counts)

# Cross-tabulation: modality x relation_category
logger.info('\n=== Modality × Relation Category ===')
cross_tab = pd.crosstab(df['modality'], df['relation_category'])
print(cross_tab)

# Dependency distance statistics
logger.info('\n=== Dependency Distance Statistics ===')
print(df['dependency_distance'].describe())

# Distance by relation category
logger.info('\n=== Mean Dependency Distance by Relation Category ===')
dist_by_rel = df.groupby('relation_category')['dependency_distance'].agg(['mean', 'median', 'std', 'count'])
print(dist_by_rel)

# Distance by modality
logger.info('\n=== Mean Dependency Distance by Modality ===')
dist_by_mod = df.groupby('modality')['dependency_distance'].agg(['mean', 'median', 'std', 'count'])
print(dist_by_mod)

## Linguistic Analysis: Dependency Distance by Modality and Relation Type

Investigate whether **spoken language exhibits shorter dependency distances** compared to written language, and whether this pattern differs by relation category (ARGUMENT, ADJUNCT, MODIFIER).

This is a core phenomenon in the dataset: spoken language often exhibits a preference for closer syntactic dependencies (dependency length minimization), while written language is more flexible.

In [ ]:
# Compute distance statistics by modality and relation category
logger.info('\n=== Distance by Modality × Relation Category ===')
dist_by_mod_rel = df.groupby(['modality', 'relation_category'])['dependency_distance'].agg(
    ['mean', 'median', 'std', 'count']
).round(2)
print(dist_by_mod_rel)

# Compute percentage of "short" arcs (distance ≤ 2) by modality
logger.info('\n=== Proportion of Short Arcs (distance ≤ 2) by Modality ===')
short_threshold = 2
short_arcs = df[df['dependency_distance'] <= short_threshold]
short_by_modality = short_arcs.groupby('modality').size() / df.groupby('modality').size()
print(short_by_modality.round(3))

# By relation category
logger.info('\n=== Proportion of Short Arcs (distance ≤ 2) by Relation Category ===')
short_by_rel = short_arcs.groupby('relation_category').size() / df.groupby('relation_category').size()
print(short_by_rel.round(3))

# By modality AND relation category
logger.info('\n=== Proportion of Short Arcs by Modality × Relation Category ===')
short_by_mod_rel = short_arcs.groupby(['modality', 'relation_category']).size() / df.groupby(['modality', 'relation_category']).size()
print(short_by_mod_rel.round(3))

## Part-of-Speech Analysis

Analyze which POS tags are most common in the dataset and how they distribute across relation categories.

In [ ]:
logger.info('\n=== Top POS Tags in Dataset ===')
upos_counts = df['upos'].value_counts().head(10)
print(upos_counts)

logger.info('\n=== Mean Dependency Distance by POS Tag (top 8) ===')
top_pos = df['upos'].value_counts().head(8).index
dist_by_pos = df[df['upos'].isin(top_pos)].groupby('upos')['dependency_distance'].agg(
    ['mean', 'median', 'count']
).round(2)
print(dist_by_pos)

## Visualization: Dependency Distance Distribution

Create visual comparisons of dependency distance distributions across modalities and relation categories.

In [ ]:
# Set style
sns.set_style('whitegrid')
sns.set_palette('Set2')

# Figure 1: Distance distribution by modality
fig, axes = plt.subplots(1, 2, figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)

# Histogram
for modality in df['modality'].unique():
    subset = df[df['modality'] == modality]['dependency_distance']
    axes[0].hist(subset, bins=30, alpha=0.6, label=modality.capitalize(), edgecolor='black')

axes[0].set_xlabel('Dependency Distance', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Dependency Distance Distribution by Modality', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].set_xlim(0, 20)  # Focus on distances 0-20

# Box plot
df.boxplot(column='dependency_distance', by='modality', ax=axes[1])
axes[1].set_xlabel('Modality', fontsize=11)
axes[1].set_ylabel('Dependency Distance', fontsize=11)
axes[1].set_title('Dependency Distance by Modality (Box Plot)', fontsize=12, fontweight='bold')
plt.suptitle('')  # Remove automatic title

plt.tight_layout()
plt.savefig('figure_1_modality_distance.png', dpi=PLOT_DPI, bbox_inches='tight')
logger.info('✓ Saved figure_1_modality_distance.png')
plt.show()

# Figure 2: Distance by relation category
fig, axes = plt.subplots(1, 2, figsize=PLOT_FIGSIZE, dpi=PLOT_DPI)

# Box plot by relation category
df.boxplot(column='dependency_distance', by='relation_category', ax=axes[0])
axes[0].set_xlabel('Relation Category', fontsize=11)
axes[0].set_ylabel('Dependency Distance', fontsize=11)
axes[0].set_title('Dependency Distance by Relation Category', fontsize=12, fontweight='bold')

# Violin plot: modality x relation_category
if len(df) > 0:
    # Create a simplified plot if dataset is small
    rel_cats = df['relation_category'].unique()
    modalities = df['modality'].unique()
    
    x_pos = 0
    colors = {'spoken': '#2ecc71', 'written': '#3498db'}
    
    for rel_cat in sorted(rel_cats):
        for i, modality in enumerate(sorted(modalities)):
            subset = df[(df['relation_category'] == rel_cat) & (df['modality'] == modality)]['dependency_distance']
            if len(subset) > 0:
                positions = [x_pos + i * 0.3]
                axes[1].boxplot(subset, positions=positions, widths=0.2, 
                               patch_artist=True,
                               boxprops=dict(facecolor=colors[modality], alpha=0.6),
                               medianprops=dict(color='red', linewidth=1.5))
        x_pos += 1.5
    
    axes[1].set_xticks([i * 1.5 + 0.15 for i in range(len(rel_cats))])
    axes[1].set_xticklabels(sorted(rel_cats))
    axes[1].set_xlabel('Relation Category', fontsize=11)
    axes[1].set_ylabel('Dependency Distance', fontsize=11)
    axes[1].set_title('Distance: Spoken (green) vs Written (blue)', fontsize=12, fontweight='bold')

plt.suptitle('')  # Remove automatic title
plt.tight_layout()
plt.savefig('figure_2_relation_category_distance.png', dpi=PLOT_DPI, bbox_inches='tight')
logger.info('✓ Saved figure_2_relation_category_distance.png')
plt.show()

## Dependency Relation Analysis

Examine which specific dependency relations are most frequent and their distance characteristics.

In [ ]:
logger.info('\n=== Top 10 Dependency Relations ===')
deprel_counts = df['deprel'].value_counts().head(10)
print(deprel_counts)

logger.info('\n=== Mean Distance by Deprel (top 10) ===')
top_deprels = df['deprel'].value_counts().head(10).index
dist_by_deprel = df[df['deprel'].isin(top_deprels)].groupby('deprel')['dependency_distance'].agg(
    ['mean', 'median', 'count']
).sort_values('mean', ascending=False).round(2)
print(dist_by_deprel)

## Key Findings Summary

This notebook has demonstrated:

1. **Dataset Structure**: Slovenian UD treebank dependency arcs classified into ARGUMENT/ADJUNCT/MODIFIER categories
2. **Data Characteristics**: 128,162 arcs (3 in mini demo), with linguistic features (deprel, POS, distance, sentence length)
3. **Morphological Richness**: Slovenian case_richness = 0.9406 (76,890/81,750 nominals marked for case)
4. **Distance Patterns**: Different relation categories show varying dependency distance distributions
5. **Modality Comparison**: Differences between spoken and written language in arc distribution and distance

**Next Steps**: To scale to the full dataset (128,162 examples), increase `MAX_EXAMPLES` or set it to `None` in the Config cell. The full dataset enables statistical testing of the dependency length minimization hypothesis across spoken vs. written language, stratified by relation category and morphological features.

In [ ]:
logger.info('\n' + '='*60)
logger.info('SUMMARY')
logger.info('='*60)
logger.info(f'Total arcs processed: {len(df)}')
logger.info(f'Relation categories: {df["relation_category"].nunique()}')
logger.info(f'Unique deprels: {df["deprel"].nunique()}')
logger.info(f'Unique POS tags: {df["upos"].nunique()}')
logger.info(f'Mean dependency distance: {df["dependency_distance"].mean():.2f}')
logger.info(f'Median dependency distance: {df["dependency_distance"].median():.2f}')
logger.info(f'Max dependency distance: {df["dependency_distance"].max()}')

if 'spoken' in df['modality'].values and 'written' in df['modality'].values:
    spoken_dist = df[df['modality'] == 'spoken']['dependency_distance'].mean()
    written_dist = df[df['modality'] == 'written']['dependency_distance'].mean()
    logger.info(f'\nSpoken mean distance: {spoken_dist:.2f}')
    logger.info(f'Written mean distance: {written_dist:.2f}')
    logger.info(f'Difference (written - spoken): {written_dist - spoken_dist:.2f}')

logger.info('\n✓ Demo notebook complete')